# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
# Local, offline setup — uses only this workspace
import json, logging, sqlite3, subprocess, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

START = Path.cwd().resolve()
ROOT = next((path for path in (START, *START.parents)
             if (path / "data" / "raw" / "flasheats.db").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root above the working directory")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from challenges.notebook_support import (
    build_order_timeline, class7_order_model, lateness_view, load_context,
    normalize_ticket_category,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
ctx = load_context()
print("Workspace:", ROOT)
print("Published pipeline gate:", ctx["quality_report"]["gate"])
RAW = ROOT / "data" / "raw"
PUBLISHED = ROOT / "data" / "processed" / "run_date=2026-09-27"

Workspace: /home/subaru/fde-work/FDE-Assignment-2
Published pipeline gate: WARN


## Load the updated FlashEats data

In [2]:
orders = ctx["orders"]                 # raw: 1,603 rows
canonical = lateness_view(ctx["journey"]) # reconciled: 1,600 business orders
restaurants = ctx["restaurants"]; drivers = ctx["drivers"]
tickets = ctx["support_tickets"]; restaurant_status = ctx["restaurant_status"]
metric_notes = ctx["metric_definitions"]
print("raw orders:", orders.shape, "canonical:", canonical.shape,
      "tickets:", tickets.shape, "restaurant_status:", restaurant_status.shape)
display(orders.head(3)); print(json.dumps(metric_notes, indent=2))

raw orders: (1603, 13) canonical: (1600, 74) tickets: (202, 5) restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order |  |  |  |
| Delivered orders have completion time |  |  |  |
| Promised ETA is valid |  |  |  |
| Event chronology is valid |  |  |  |
| “Late” has an agreed definition |  |  |  |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

---

### Completed answer

| Business assumption | Data expectation/test | Result | Severity |
|---|---|---|---|
| One row = one order | unique `order_id` | 1,603 rows / 1,600 IDs; 3 traffic-only conflicts | WARN |
| Delivered has completion | delivered ⇒ actual time | 37 canonical delivered orders missing actual time | WARN |
| Promised ETA valid | promised ≥ created | 4 raw rows violate | WARN |
| Chronology valid | pickup ≤ delivery | 5 raw rows violate | WARN |
| “Late” agreed | documented owner/threshold/denominator | conflicting definitions; no owner | UNKNOWN |

The claim is numerically reproducible only after an explicit contract. The three duplicated IDs are **not resolved by keeping the first traffic value**: both rows are retained in lineage and traffic becomes unknown.


In [3]:
# Challenge 1 — executable contract checks on raw and canonical grains
for column in ["created_at","promised_eta","pickup_at","actual_delivery_at"]:
    orders[column + "_parsed"] = pd.to_datetime(orders[column], format="mixed", errors="coerce")
duplicate_rows = orders[orders.order_id.duplicated(keep=False)].sort_values("order_id")
contract = pd.DataFrame([
    ["Business grain", "unique order_id", len(orders)-orders.order_id.nunique(), "WARN"],
    ["Delivered completion", "delivered has actual time", int((canonical.final_status.eq("delivered") & canonical.actual_delivery_at.isna()).sum()), "WARN"],
    ["Promised ETA chronology", "promised >= created", int((orders.promised_eta_parsed < orders.created_at_parsed).sum()), "WARN"],
    ["Pickup/delivery chronology", "pickup <= delivery", int((orders.pickup_at_parsed > orders.actual_delivery_at_parsed).sum()), "WARN"],
    ["KPI definition", "one approved owner/contract", 1, "UNKNOWN"],
], columns=["Check","Expectation","Observed exceptions","Status"])
display(contract); display(duplicate_rows[["order_id","traffic_bucket"]])
print("Documented reconciliation:", ctx["cleaning_report"]["sources"]["orders"]["actions"])

,Check,Expectation,Observed exceptions,Status
0,Business grain,unique order_id,3,WARN
1,Delivered completion,delivered has actual time,37,WARN
2,Promised ETA chronology,promised >= created,4,WARN
3,Pickup/delivery chronology,pickup <= delivery,5,WARN
4,KPI definition,one approved owner/contract,1,UNKNOWN


,order_id,traffic_bucket
119,O00120,severe
1600,O00120,medium
722,O00723,medium
1601,O00723,high
1301,O01302,medium
1602,O01302,high


Documented reconciliation: [{'action': 'reconcile_noncritical_traffic_conflict', 'evidence': [{'differing_columns': ['traffic_bucket'], 'order_id': 'O00120', 'source_rows': [119, 1600], 'traffic_values': ['severe', 'medium']}, {'differing_columns': ['traffic_bucket'], 'order_id': 'O00723', 'source_rows': [722, 1601], 'traffic_values': ['medium', 'high']}, {'differing_columns': ['traffic_bucket'], 'order_id': 'O01302', 'source_rows': [1301, 1602], 'traffic_values': ['medium', 'high']}], 'reason': 'Every other raw order column agrees; traffic is unknown, not a category', 'reconciled_order_ids': ['O00120', 'O00723', 'O01302'], 'rows': 3, 'unresolved_traffic_classifications': 3}, {'action': 'normalize_known_status', 'reason': 'Case/whitespace variant of observed delivered or cancelled', 'rows': 5}]


# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

---

### Completed answer

| Definition | Numerator / denominator | Late rate | Business meaning |
|---|---:|---:|---|
| VP Operations: delivered + comparable timestamps, delay > 0 | 843 / 1,495 | 56.39% | promise adherence |
| Support: same eligibility, delay > 10 min | 349 / 1,495 | 23.34% | materially late experience |
| Any delay / all canonical orders | 843 / 1,600 | 52.69% | mixes cancellations/missing outcomes into denominator |
| Historical delivered/non-null population | 843 / 1,495 | 56.39% | reproduces rounded “56%” |

Leadership should not publish an unlabeled 56%. If a provisional metric is necessary, publish **56.39% (843/1,495), delivered orders with comparable promised/actual timestamps, >0 minutes, 37 delivered missing completion and 68 cancellations excluded**, marked unapproved. A formally named KPI owner—jointly informed by Operations, Support and Finance—must approve threshold, denominator and cancellation/refund handling.


In [4]:
# Challenge 2 — same canonical grain, explicit numerators and denominators
eligible = canonical[canonical.ldr_eligible]
definition_table = pd.DataFrame([
    ["VP Ops: delivered/comparable, >0 min", int(eligible.late.sum()), len(eligible), eligible.late.mean(), "Promise adherence"],
    ["Support: delivered/comparable, >10 min", int(eligible.late_10.sum()), len(eligible), eligible.late_10.mean(), "Meaningfully late"],
    ["Any delay / all canonical orders", int(canonical.late.sum()), len(canonical), canonical.late.mean(), "Mixes in non-eligible orders"],
    ["Historical delivered/non-null", int(eligible.late.sum()), len(eligible), eligible.late.mean(), "Reproduces rounded claim"],
], columns=["Definition","Late","Denominator","Rate","Business meaning"])
display(definition_table.round({"Rate": 4}))

,Definition,Late,Denominator,Rate,Business meaning
0,"VP Ops: delivered/comparable, >0 min",843,1495,0.5639,Promise adherence
1,"Support: delivered/comparable, >10 min",349,1495,0.2334,Meaningfully late
2,Any delay / all canonical orders,843,1600,0.5269,Mixes in non-eligible orders
3,Historical delivered/non-null,843,1495,0.5639,Reproduces rounded claim


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

---

### Completed answer

Safe representation normalization includes trim/lowercase for `Delivered`, `HIGH`, `READY`, `Ready`, `ready `, `Late Delivery`, and `late_delivery `. It does **not** merge `handoff` with `handed_off`, `unknown` with a known state, or `ETA issue` with `eta_changed` without owner confirmation.

Observed raw counts include order status `delivered` 1,530 / `Delivered` 5 / `cancelled` 68; traffic `medium` 635 / `high` 448 / `low` 381 / `severe` 136 / `HIGH` 3; restaurant status includes 171 preparing, 168 handed_off, 157 ready plus representation variants and one each handoff/unknown. Ticket categories retain seven substantive concepts plus representation variants.


In [5]:
# Challenge 3 — inspect raw vocabularies and document only safe transformations
for label, series in {
    "final_status": orders.final_status, "traffic_bucket": orders.traffic_bucket,
    "restaurant_status": restaurant_status.status, "support_category": tickets.category,
}.items():
    print("\n", label); display(series.value_counts(dropna=False).rename("count").to_frame())
normalization_decisions = pd.DataFrame([
    ["Delivered → delivered", "Safe", "case-only representation"],
    ["HIGH → high", "Safe", "case-only representation"],
    ["READY/Ready/ready  → ready", "Safe", "case/whitespace only"],
    ["Late Delivery/late_delivery  → late_delivery", "Safe", "case/space representation"],
    ["handoff → handed_off", "Owner confirmation", "may denote action vs completed state"],
    ["ETA issue → eta_changed", "Owner confirmation", "broader issue is not necessarily a change"],
    ["unknown → any status", "Unsafe", "unknown is missing semantics"],
], columns=["Candidate mapping","Decision","Reason"])
display(normalization_decisions)


 final_status


,count
final_status,
delivered,1530
cancelled,68
Delivered,5



 traffic_bucket


,count
traffic_bucket,
medium,635
high,448
low,381
severe,136
HIGH,3



 restaurant_status


,count
status,
preparing,171
handed_off,168
ready,157
ready,2
READY,1
Ready,1
handoff,1
unknown,1



 support_category


,count
category,
late_delivery,38
eta_changed,37
restaurant_delay,37
ready_but_waiting,33
status_mismatch,29
driver_not_moving,25
Late Delivery,1
late_delivery,1
ETA issue,1


,Candidate mapping,Decision,Reason
0,Delivered → delivered,Safe,case-only representation
1,HIGH → high,Safe,case-only representation
2,READY/Ready/ready → ready,Safe,case/whitespace only
3,Late Delivery/late_delivery → late_delivery,Safe,case/space representation
4,handoff → handed_off,Owner confirmation,may denote action vs completed state
5,ETA issue → eta_changed,Owner confirmation,broader issue is not necessarily a change
6,unknown → any status,Unsafe,unknown is missing semantics


# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

---

### Completed answer

| Relationship | Matched / all rows | Coverage | Status / risk |
|---|---:|---:|---|
| orders.restaurant_id → restaurants | 1,597 / 1,600 | 99.81% | WARN: 3 null IDs |
| orders.driver_id → drivers | 1,597 / 1,600 | 99.81% | WARN: 3 null IDs |
| tickets.order_id → orders | 199 / 202 | 98.51% | WARN: 3 missing order IDs |
| restaurant_status.order_id → orders | 502 / 502 | 100.00% | PASS |

Coverage among non-null keys is 100% for all four relationships. A 1% gap may be tolerable for exploratory weekly aggregates if quantified, but not for driver/restaurant accountability, support case resolution or live intervention when the missing rows are the target population.


In [6]:
# Challenge 4 — report all-row and non-null-key coverage separately
relationships = []
for label, child, key, parent_ids in [
    ("orders.restaurant_id → restaurants", canonical, "restaurant_id", set(restaurants.restaurant_id)),
    ("orders.driver_id → drivers", canonical, "driver_id", set(drivers.driver_id)),
    ("tickets.order_id → orders", tickets, "order_id", set(canonical.order_id)),
    ("restaurant_status.order_id → orders", restaurant_status, "order_id", set(canonical.order_id)),
]:
    non_null = child[key].notna(); matched = non_null & child[key].isin(parent_ids)
    relationships.append([label, len(child), int(non_null.sum()), int(matched.sum()),
        matched.mean(), matched.sum()/non_null.sum(), int((~non_null).sum()), int((non_null & ~matched).sum())])
coverage = pd.DataFrame(relationships, columns=["Relationship","Rows","Non-null","Matched","All-row coverage","Non-null coverage","Missing key","Unmapped non-null"])
display(coverage.round({"All-row coverage": 4, "Non-null coverage": 4}))

,Relationship,Rows,Non-null,Matched,All-row coverage,Non-null coverage,Missing key,Unmapped non-null
0,orders.restaurant_id → restaurants,1600,1597,1597,0.9981,1.0,3,0
1,orders.driver_id → drivers,1600,1597,1597,0.9981,1.0,3,0
2,tickets.order_id → orders,202,199,199,0.9851,1.0,3,0
3,restaurant_status.order_id → orders,502,502,502,1.0000,1.0,0,0


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

---

### Completed answer

There are **502 status rows for 500/1,600 orders (31.25% order coverage)**, with up to two records per order. Six updates predate order creation; none occur after an observed delivery; 336 are at/before pickup. For comparable delivered rows, the median status-to-delivery gap is **53.77 min** (90th percentile absolute gap **78.52 min**).

- **Weekly analytics:** WARN/usable only with the 31.25% coverage caveat and six invalid records excluded/flagged.
- **Live customer ETA:** FAIL; sparse event-driven snapshots and ~54-minute median gap to delivery do not establish a live-freshness SLA.
- **Restaurant accountability:** FAIL; partial coverage, ambiguous `handoff` semantics and pre-creation records cannot support fair attribution.

Freshness needs an owner-defined maximum age at a decision timestamp; this export alone provides no extraction/snapshot timestamp SLA.


In [7]:
# Challenge 5 — relate status timestamps to canonical order lifecycle
status_timing = restaurant_status.merge(canonical[["order_id","created_at","pickup_at","actual_delivery_at"]],
                                        on="order_id", how="left", validate="many_to_one")
for column in ["last_updated_at","created_at","pickup_at","actual_delivery_at"]:
    status_timing[column] = pd.to_datetime(status_timing[column], format="mixed", errors="coerce")
status_timing["update_to_delivery_min"] = (status_timing.actual_delivery_at-status_timing.last_updated_at).dt.total_seconds()/60
freshness = pd.DataFrame([{
    "status_rows":len(status_timing), "orders_covered":status_timing.order_id.nunique(),
    "order_coverage":status_timing.order_id.nunique()/len(canonical),
    "before_created":(status_timing.last_updated_at < status_timing.created_at).sum(),
    "after_delivery":(status_timing.actual_delivery_at.notna() & (status_timing.last_updated_at > status_timing.actual_delivery_at)).sum(),
    "at_or_before_pickup":(status_timing.pickup_at.notna() & (status_timing.last_updated_at <= status_timing.pickup_at)).sum(),
    "median_update_to_delivery_min":status_timing.update_to_delivery_min.median(),
    "p90_abs_update_to_delivery_min":status_timing.update_to_delivery_min.abs().quantile(.9),
}])
display(freshness.round({"order_coverage": 4, "median_update_to_delivery_min": 2, "p90_abs_update_to_delivery_min": 2}))

,status_rows,orders_covered,order_coverage,before_created,after_delivery,at_or_before_pickup,median_update_to_delivery_min,p90_abs_update_to_delivery_min
0,502,500,0.3125,6,0,336,53.77,78.52


# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

---

### Completed answer

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | WARN | 1,603 rows / 1,600 IDs; 3 traffic-only conflicts | use documented unknown-traffic reconciliation |
| Timestamp chronology | WARN | 4 promised-before-created; 5 pickup-after-delivery; 37 missing completions | fix producers; exclude/flag per contract |
| KPI definition | UNKNOWN | stakeholders disagree; no named owner | approve threshold/denominator/refund policy |
| Category semantics | WARN | representation variants plus semantic ambiguity | normalize representation only; owner maps semantics |
| Cross-source mapping | WARN | non-null keys map 100%, but 3 null dimension IDs and 3 ticket order IDs | remediate missing keys based on use case |
| Freshness | FAIL | restaurant status covers 31.25%; no live SLA | instrument governed status events/freshness |

**Do not publish bare “LDR = 56%” today.** The rounded number is reproducible, but the data gate is WARN and the semantic contract is UNKNOWN. Publish only a clearly labelled provisional 56.39% with numerator, denominator and exclusions, or wait for owner approval and remediation.


In [8]:
# Challenge 6 — explicit decision gate
validation_report = pd.DataFrame([
    ["Business grain","WARN","3 traffic-only duplicate IDs","Use documented reconciliation; traffic unknown"],
    ["Timestamp chronology","WARN","4 promised-before-created; 5 pickup-after-delivery; 37 missing completion","Fix producers; flag/exclude by contract"],
    ["KPI definition","UNKNOWN","Conflicting stakeholder definitions; no owner","Name owner and approve contract"],
    ["Category semantics","WARN","Representation and semantic variants","Normalize representation only"],
    ["Cross-source mapping","WARN","100% non-null mapping; six relevant null keys","Remediate based on decision risk"],
    ["Freshness","FAIL","31.25% status coverage; no live SLA","Instrument governed status stream"],
], columns=["Check","Status","Evidence","Action"])
display(validation_report)
publish_56_percent = "NO — only a labelled provisional 56.39% (843/1,495) is defensible"
print(publish_56_percent)

,Check,Status,Evidence,Action
0,Business grain,WARN,3 traffic-only duplicate IDs,Use documented reconciliation; traffic unknown
1,Timestamp chronology,WARN,4 promised-before-created; 5 pickup-after-delivery; 37 missing completion,Fix producers; flag/exclude by contract
2,KPI definition,UNKNOWN,Conflicting stakeholder definitions; no owner,Name owner and approve contract
3,Category semantics,WARN,Representation and semantic variants,Normalize representation only
4,Cross-source mapping,WARN,100% non-null mapping; six relevant null keys,Remediate based on decision risk
5,Freshness,FAIL,31.25% status coverage; no live SLA,Instrument governed status stream


NO — only a labelled provisional 56.39% (843/1,495) is defensible


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**